# Part 2: Matching sales to EPCs by address

Joins Price Paid Data to EPCs on address. Tries exact matching on address word sets first, then a subset test, which raised the usable match rate from 63.7% to 67.2%. Also looks at why sales fail to match, and at ambiguous matches.

Findings and decisions: `logs/part_2_regex_parsing.md`

#### Load Price data

In [186]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)


folder = Path('../data/raw')
file_gen = folder.glob('ppd_data*.csv')
pp_raw = pd.concat([pd.read_csv(f) for f in file_gen], ignore_index=True)
pp_raw['deed_date'] = pp_raw['deed_date'].astype('datetime64[ns]')
pp_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 135557 entries, 0 to 135556
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   unique_id             135557 non-null  str           
 1   price_paid            135557 non-null  int64         
 2   deed_date             135557 non-null  datetime64[ns]
 3   postcode              135115 non-null  str           
 4   property_type         135557 non-null  str           
 5   new_build             135557 non-null  str           
 6   estate_type           135557 non-null  str           
 7   saon                  11820 non-null   str           
 8   paon                  135557 non-null  str           
 9   street                132302 non-null  str           
 10  locality              57572 non-null   str           
 11  town                  135557 non-null  str           
 12  district              135557 non-null  str           
 13  county    

#### Load EPC data

In [187]:
epc_raw = pd.read_csv('../data/raw/EDC_OXFORDSHIRE_2015_2025.csv', usecols= [
    'certificate_number', 'address1', 'address2', 'address3', 'postcode',
    'posttown', 'address', 'local_authority', 'local_authority_label',
    'built_form', 'construction_age_band', 'current_energy_efficiency',
    'current_energy_rating', 'energy_consumption_current', 'extension_count', 'floor_height', 'heating_cost_current',
    'inspection_date', 'lodgement_date', 'number_habitable_rooms',
    'number_heated_rooms', 'potential_energy_efficiency', 'property_type',
    'tenure', 'total_floor_area', 'transaction_type', 'uprn', 'uprn_source'
])
epc_raw['inspection_date'] = epc_raw['inspection_date'].astype('datetime64[ns]')
epc_raw['lodgement_date'] = epc_raw['lodgement_date'].astype('datetime64[ns]')
epc_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 216054 entries, 0 to 216053
Data columns (total 28 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   certificate_number           216054 non-null  str           
 1   address1                     216054 non-null  str           
 2   address2                     117532 non-null  str           
 3   address3                     27304 non-null   str           
 4   postcode                     216054 non-null  str           
 5   posttown                     213327 non-null  str           
 6   address                      216054 non-null  str           
 7   local_authority              216054 non-null  str           
 8   local_authority_label        216054 non-null  str           
 9   built_form                   213548 non-null  str           
 10  construction_age_band        216054 non-null  str           
 11  current_energy_efficiency    216054 n

In [188]:
print(pp_raw[['postcode','saon', 'paon', 'street', 'locality', 'town']].head())
print(epc_raw[['postcode', 'address3', 'address2', 'address1', 'posttown']].head())

   postcode           saon                  paon street   locality       town
0  HP18 9UH   PARK COTTAGE   CHILLING PLACE FARM    NaN      BRILL  AYLESBURY
1  HP18 9UY            NaN         OAKCROFT FARM    NaN  BOARSTALL  AYLESBURY
2  HP18 9UZ            NaN         TUTHER CORNER    NaN      BRILL  AYLESBURY
3  HP18 9UZ            NaN         TUTHER CORNER    NaN      BRILL  AYLESBURY
4  HP18 9XA  HEADLEY HOUSE  PIDDINGTON GATE FARM    NaN      BRILL  AYLESBURY
   postcode address3                address2                  address1         posttown
0   OX4 3UJ      NaN                     NaN            10 Newman Road           OXFORD
1  OX33 1NL      NaN                Wheatley           10 Barlow Close           OXFORD
2   OX4 3PG      NaN                     NaN          65 Campbell Road           OXFORD
3   OX7 6BE      NaN  Shipton-Under-Wychwood     12 St. Michaels Close  CHIPPING NORTON
4  OX16 5BG      NaN            Tramway Road  Flat 10, Canalside House          BANBURY


#### Comparing address data

In [189]:
print(pp_raw[pp_raw['postcode'] == 'HP18 9UH'][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].head())
print(epc_raw[epc_raw['postcode'] == 'HP18 9UH'][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].head())

   postcode locality street          saon                 paon       town
0  HP18 9UH    BRILL    NaN  PARK COTTAGE  CHILLING PLACE FARM  AYLESBURY
       postcode address3             address2                       address1   posttown                                   address
27420  HP18 9UH      NaN                Brill  Oak Lodge Chilling Place Farm  AYLESBURY      Oak Lodge Chilling Place Farm, Brill
60325  HP18 9UH    Brill  Chilling Place Farm                   Park Cottage  AYLESBURY  Park Cottage, Chilling Place Farm, Brill


In [190]:
print(pp_raw[pp_raw['postcode'] == 'OX12 9RG'][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].sort_values('saon', ascending=True).head(10))
print(epc_raw[epc_raw['postcode'] == 'OX12 9RG'][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].sort_values('address1', ascending=True).head(10))

       postcode        locality        street saon        paon     town
92129  OX12 9RG  LETCOMBE REGIS  SOUTH STREET    1  BROOK VIEW  WANTAGE
92130  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   10  BROOK VIEW  WANTAGE
92131  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   11  BROOK VIEW  WANTAGE
92132  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   11  BROOK VIEW  WANTAGE
92133  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   13  BROOK VIEW  WANTAGE
92134  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   13  BROOK VIEW  WANTAGE
92135  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   15  BROOK VIEW  WANTAGE
92136  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   15  BROOK VIEW  WANTAGE
92137  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   16  BROOK VIEW  WANTAGE
92138  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   17  BROOK VIEW  WANTAGE
        postcode        address3      address2       address1 posttown                                      address
205645  OX12 9RG  Letcombe Regis  South Street   1 Brook View  WANTAGE   1 Brook View, South

- Comparison of address data
    - PP full caps vs EPC capital case
    - pp_'locality' is typically epc_'address3' but can be epc_'address2', depending on how the address is split
    - pp_'saon' or pp_'saon'+pp_'paon' can be epc_'address1'
    - pp_'town' == epc_'post_town'
    - Both appear use 'FLAT 8' vs 'Flat 8'

In [191]:
print(pp_raw[pp_raw['postcode'] == 'MK18 4AS'][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].sort_values('saon', ascending=False).head(10))
print(epc_raw[epc_raw['postcode'] == 'MK18 4AS'][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].sort_values('address1', ascending=False).head(10))

    postcode locality        street saon             paon        town
41  MK18 4AS  FINMERE  FULWELL ROAD  NaN      FIVE GABLES  BUCKINGHAM
42  MK18 4AS  FINMERE  FULWELL ROAD  NaN       HEDGELANDS  BUCKINGHAM
43  MK18 4AS  FINMERE  FULWELL ROAD  NaN        HIGH ACRE  BUCKINGHAM
44  MK18 4AS  FINMERE  FULWELL ROAD  NaN        OAK HOUSE  BUCKINGHAM
45  MK18 4AS  FINMERE  FULWELL ROAD  NaN        OAK HOUSE  BUCKINGHAM
46  MK18 4AS  FINMERE  FULWELL ROAD  NaN  STONEWELL HOUSE  BUCKINGHAM
47  MK18 4AS  FINMERE  FULWELL ROAD  NaN   SYCAMORE HOUSE  BUCKINGHAM
48  MK18 4AS  FINMERE  FULWELL ROAD  NaN   SYCAMORE HOUSE  BUCKINGHAM
49  MK18 4AS  FINMERE  FULWELL ROAD  NaN   SYCAMORE HOUSE  BUCKINGHAM
50  MK18 4AS  FINMERE  FULWELL ROAD  NaN        THE HAVEN  BUCKINGHAM
        postcode address3      address2         address1    posttown                                address
175699  MK18 4AS  Finmere  Fulwell Road      Top Gardens  BUCKINGHAM     Top Gardens, Fulwell Road, Finmere
53071   MK18 4

- Matches well on postcode and paon = address1, saon is null
- One record where posttown uses the value put in address3 for the other records

In [192]:
print(pp_raw[pp_raw['postcode'] == 'OX15 0PT'][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].sort_values('paon', ascending=False).head(10))
print(epc_raw[epc_raw['postcode'] == 'OX15 0PT'][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].sort_values('address1', ascending=False).head(10))

     postcode locality        street saon                paon     town
300  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN         THE WILLOWS  BANBURY
298  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN       ORCHARD HOUSE  BANBURY
299  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN       ORCHARD HOUSE  BANBURY
297  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN  OLD WESLYAN CHAPEL  BANBURY
296  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN           HOMELANDS  BANBURY
295  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN        HADLEY HOUSE  BANBURY
294  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN       BAMFORD HOUSE  BANBURY
293  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN       APPLE COTTAGE  BANBURY
302  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN                   5  BANBURY
301  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN                   4  BANBURY
        postcode address3      address2            address1 posttown                                    address
130406  OX15 0PT  Clifton  Chapel Close       The Homestead  BANBURY       The Homestead, Chapel Close, Cli

#### Cleaning

Fix trailing whitespace

In [193]:
pp = pp_raw.copy()

for c in pp.select_dtypes(include=['string']).columns:
    pp[c] = pp[c].str.strip()
    print(f"{c} stripped")

unique_id stripped
postcode stripped
property_type stripped
new_build stripped
estate_type stripped
saon stripped
paon stripped
street stripped
locality stripped
town stripped
district stripped
county stripped
transaction_category stripped
linked_data_uri stripped


In [194]:
epc = epc_raw.copy()

for c in epc.select_dtypes(include=['string']).columns:
    epc[c] = epc[c].str.strip()
    print(f"{c} stripped")

certificate_number stripped
address1 stripped
address2 stripped
address3 stripped
postcode stripped
posttown stripped
address stripped
local_authority stripped
local_authority_label stripped
built_form stripped
construction_age_band stripped
current_energy_rating stripped
property_type stripped
tenure stripped
transaction_type stripped
uprn_source stripped


Remove commas

In [195]:
epc['address1'] = epc['address1'].str.replace(',', '', regex=False)

- For loops over columns are fast (over rows are slow). List comprehensions wont work as they lose the index.

#### Initial join (exact token matches)

Generate combined addresses (space separated) and match cases

In [196]:
pp['address'] = pp['paon'].str.cat([pp['saon'], pp['street'], pp['locality'], pp['postcode']], sep=' ', na_rep='')
print(pp['address'])
epc['address'] = epc['address1'].str.cat([epc['address2'], epc['address3'], epc['postcode']], sep=' ', na_rep='').apply(str.upper)
print(epc['address'])

0          CHILLING PLACE FARM PARK COTTAGE  BRILL HP18 9UH
1                        OAKCROFT FARM   BOARSTALL HP18 9UY
2                            TUTHER CORNER   BRILL HP18 9UZ
3                            TUTHER CORNER   BRILL HP18 9UZ
4         PIDDINGTON GATE FARM HEADLEY HOUSE  BRILL HP18...
                                ...                        
135552                   THE BUTCHERS GROUND   NORTH LEIGH 
135553                  TURLEY FARM LOTS 2 3 AND 4  HAILEY 
135554                        TURLEY LANE PLOT 154  HAILEY 
135555                       PLOTS 79-84   LONG HANBOROUGH 
135556                                114 GARAGE NEW ROAD  
Name: address, Length: 135557, dtype: str
0                                  10 NEWMAN ROAD   OX4 3UJ
1                        10 BARLOW CLOSE WHEATLEY  OX33 1NL
2                                65 CAMPBELL ROAD   OX4 3PG
3         12 ST. MICHAELS CLOSE SHIPTON-UNDER-WYCHWOOD  ...
4            FLAT 10 CANALSIDE HOUSE TRAMWAY ROAD  OX16 5B

- The space separator is being added before/after null values (even though they are empty strings)
- Some commas are making it into the combined addresses

In [197]:
pp['address_tokens'] = pp['address'].str.split().apply(frozenset)
epc['address_tokens'] = epc['address'].str.split().apply(frozenset)
print(epc['address_tokens'])

0                   frozenset({OX4, ROAD, NEWMAN, 10, 3UJ})
1         frozenset({BARLOW, CLOSE, 10, WHEATLEY, 1NL, O...
2                 frozenset({65, OX4, ROAD, 3PG, CAMPBELL})
3         frozenset({6BE, 12, CLOSE, OX7, ST., SHIPTON-U...
4         frozenset({HOUSE, TRAMWAY, CANALSIDE, ROAD, OX...
                                ...                        
216049                frozenset({OX12, 30, APPLETONS, 7GG})
216050        frozenset({ROAD, 1D, 6ED, OX2, WALTON, WELL})
216051             frozenset({ROAD, 9EJ, OX2, SYCAMORE, 8})
216052         frozenset({OX4, CRESCENT, SPENCER, 4SW, 68})
216053    frozenset({1FQ, 21, CLOSE, WHEATLEY, OCHRE, OX...
Name: address_tokens, Length: 216054, dtype: object


- Additional spaces are removed during split
- Use .apply(set) rather than set() so that the function is applied to each row of the new series, rather than trying to apply it to the entire series at once

In [198]:
first_join = pp.merge(epc, on='address_tokens', how='inner')

print(f"""New table has {first_join.shape[0]} rows, but this includes multiple EPC records per property matched. 
      Matched {first_join['unique_id'].nunique()} Price paid records with an EPC certificate. Match rate: {100* first_join['unique_id'].nunique()/pp.shape[0]:.1f}%""")


New table has 107935 rows, but this includes multiple EPC records per property matched. 
      Matched 96461 Price paid records with an EPC certificate. Match rate: 71.2%


In [199]:
print(first_join.sort_index(axis=1).sample(20))

                      address1     address2  address3                                     address_tokens                                address_x                                address_y     built_form        certificate_number           construction_age_band       county  current_energy_efficiency current_energy_rating  deed_date             district  energy_consumption_current estate_type  extension_count  floor_height  heating_cost_current inspection_date                                    linked_data_uri local_authority local_authority_label    locality lodgement_date new_build  number_habitable_rooms  number_heated_rooms     paon postcode_x postcode_y          posttown  potential_energy_efficiency  price_paid property_type_x property_type_y saon               street          tenure  total_floor_area              town transaction_category transaction_type                             unique_id          uprn      uprn_source
70391           1 Oat Crescent        Grove       NaN    fr

#### Second join

In [200]:
epc_address_cols = ['address1', 'address2', 'address3', 'postcode']
pp_address_cols = ['paon', 'saon', 'street', 'locality', 'postcode']

print(f"Commas replaced in epc:\n{epc[epc_address_cols].apply(lambda col: col.str.count(',')).sum()}")
print(f"Commas replaced in pp:\n{pp[pp_address_cols].apply(lambda col: col.str.count(',')).sum()}")

for c in epc_address_cols:
    epc[c] = epc[c].str.replace(',', '', regex=False)
for c in pp_address_cols:
    pp[c] = pp[c].str.replace(',', '', regex=False)

Commas replaced in epc:
address1       0.0
address2    3910.0
address3     137.0
postcode       0.0
dtype: float64
Commas replaced in pp:
paon        4356.0
saon          33.0
street         0.0
locality       0.0
postcode       0.0
dtype: float64


In [201]:
pp['address'] = pp['paon'].str.cat([pp['saon'], pp['street'], pp['locality'], pp['postcode']], sep=' ', na_rep='')
epc['address'] = epc['address1'].str.cat([epc['address2'], epc['address3'], epc['postcode']], sep=' ', na_rep='').apply(str.upper)

pp['address_tokens'] = pp['address'].str.split().apply(frozenset)
epc['address_tokens'] = epc['address'].str.split().apply(frozenset)

second_join = pp.merge(epc, on='address_tokens', how='inner')

print(f"""New table has {second_join.shape[0]} rows, but this includes multiple EPC records per property matched. 
      Matched {second_join['unique_id'].nunique()} unique Price paid records with an EPC certificate. Match rate: {100* second_join['unique_id'].nunique()/pp.shape[0]:.1f}%""")


New table has 110478 rows, but this includes multiple EPC records per property matched. 
      Matched 98784 unique Price paid records with an EPC certificate. Match rate: 72.9%


Filtering EPC matches

In [202]:
second_join['time_to_sale'] = second_join['deed_date'] - second_join['inspection_date']

pre_epc = second_join[second_join['time_to_sale'] >= '0 days']
print(f"Properties matched with EPCs only before sale: {pre_epc.shape[0]}")

late_epc = second_join[second_join['time_to_sale'] < '0 days']
print(f"Properties matched with EPCs after sale: {late_epc.shape[0]}")

Properties matched with EPCs only before sale: 91240
Properties matched with EPCs after sale: 19238


In [203]:
dup_rows = second_join[['unique_id', 'certificate_number']].groupby('unique_id')['certificate_number'].agg('count')
print(f"Properties with more than 1 EPC: {(dup_rows > 1).sum()}. Rows to remove: {second_join.shape[0] - second_join['unique_id'].nunique()}")

Properties with more than 1 EPC: 10520. Rows to remove: 11694


Filtered to properties with EPC after sale and drop all but closest in date

In [204]:
filtered_out = second_join[second_join['time_to_sale'] < '0 days'].sort_values(by='time_to_sale', ascending=False).drop_duplicates(subset='unique_id')
print(filtered_out.shape[0])

print(filtered_out['time_to_sale'].describe())

18140
count                             18140
mean     -1547 days +01:05:00.859977952
std        1072 days 05:22:37.838681232
min                -3971 days +00:00:00
25%                -2356 days +18:00:00
50%                -1418 days +00:00:00
75%                 -630 days +00:00:00
max                   -1 days +00:00:00
Name: time_to_sale, dtype: object


- median days after sale is 1418 day (ca. 4 years), 75th percentile of 630 days.
- *Decision: Drop all properties with EPCs after sale. Return to filter more specifically if time allows.*

In [205]:
drop_neg = second_join[second_join['time_to_sale'] >= '0 days']
second_join_presale_epc = drop_neg.sort_values(by='time_to_sale', ascending=True).drop_duplicates(subset='unique_id')
print(f"Properties remaining with matched pre-sale EPCs: {second_join_presale_epc.shape[0]}")
print(f"Properties matched but with no usable pre-sale EPC: {second_join['unique_id'].nunique() - second_join_presale_epc.shape[0]}")

Properties remaining with matched pre-sale EPCs: 86358
Properties matched but with no usable pre-sale EPC: 12426


#### Sampling unmatched records

In [206]:
unmatched_pp = pp.merge(epc, on='address_tokens', how='left_anti')

unmatched_epc = pp.merge(epc, on='address_tokens', how='right_anti')

epc_address_cols = ['address_y', 'address1', 'address2', 'address3', 'postcode_y']
pp_address_cols = ['address_x', 'paon', 'saon', 'street', 'locality', 'postcode_x', 'deed_date']

print(unmatched_pp[pp_address_cols].sample(20))
print(unmatched_epc[epc_address_cols].sample(20))

                                                address_x                paon     saon              street               locality postcode_x  deed_date
132538  WINDRUSH COURT 67 FLAT 50 ST MARYS MEAD  OX28 4FD   WINDRUSH COURT 67  FLAT 50       ST MARYS MEAD                    NaN   OX28 4FD 2016-11-11
71180              67  LITTLEWORTH ROAD WHEATLEY OX33 1NW                  67      NaN    LITTLEWORTH ROAD               WHEATLEY   OX33 1NW 2018-12-07
5693                        18  EDINBURGH CLOSE  OX16 0NY                  18      NaN     EDINBURGH CLOSE                    NaN   OX16 0NY 2020-12-01
115788                   64  MEADOW CLOSE FARMOOR OX2 9NZ                  64      NaN        MEADOW CLOSE                FARMOOR    OX2 9NZ 2018-10-19
146798  STONE LEIGH  MANOR ROAD SANDFORD ST MARTIN OX7...         STONE LEIGH      NaN          MANOR ROAD     SANDFORD ST MARTIN    OX7 7AG 2018-07-16
117546                  8  FORTY GREEN SHRIVENHAM SN6 8GU                   8      NaN  

In [207]:
test_pp = unmatched_pp[pp_address_cols]
test_epc = unmatched_epc[epc_address_cols]

postcode_check = test_pp.merge(test_epc, left_on='postcode_x', right_on='postcode_y', how='inner') 

In [208]:
print(postcode_check.shape[0])

410977


In [209]:
print('Null in price paid set:')
print(pp.isna().mean())
print()
print('Null in unmatched price paid set:')
print(test_pp.isna().mean())


Null in price paid set:
unique_id               0.000000
price_paid              0.000000
deed_date               0.000000
postcode                0.003261
property_type           0.000000
new_build               0.000000
estate_type             0.000000
saon                    0.912804
paon                    0.000000
street                  0.024012
locality                0.575293
town                    0.000000
district                0.000000
county                  0.000000
transaction_category    0.000000
linked_data_uri         0.000000
address                 0.000000
address_tokens          0.000000
dtype: float64

Null in unmatched price paid set:
address_x     0.000000
paon          0.000000
saon          0.871346
street        0.038642
locality      0.525712
postcode_x    0.012020
deed_date     0.000000
dtype: float64


- non-null saon is enriched in unmatched set

In [210]:
unmatched_postcodes = pp.merge(epc, on='postcode', how='left_anti')
print(unmatched_postcodes['unique_id'].nunique())

2269


In [211]:
print(pp['deed_date'].describe())
print(unmatched_pp['deed_date'].describe())

count                           135557
mean     2020-07-01 14:47:52.680865024
min                2015-01-02 00:00:00
25%                2017-10-26 00:00:00
50%                2020-10-09 00:00:00
75%                2022-12-21 00:00:00
max                2025-12-23 00:00:00
Name: deed_date, dtype: object
count                            36773
mean     2019-12-06 14:45:30.372827904
min                2015-01-02 00:00:00
25%                2017-03-31 00:00:00
50%                2020-01-28 00:00:00
75%                2022-02-25 00:00:00
max                2025-12-23 00:00:00
Name: deed_date, dtype: object


- deed_date distribution is similar in full PP set and unmatched

In [212]:
print(pd.concat([pp['new_build'].value_counts(normalize=True), unmatched_pp['new_build'].value_counts(normalize=True)], axis=1, keys=['full_pp', 'unmatched_pp']))

            full_pp  unmatched_pp
new_build                        
N          0.819877      0.899165
Y          0.180123      0.100835


- Unmatched are slightly less likely to be new builds

In [213]:
print(pd.concat([pp['estate_type'].value_counts(normalize=True), unmatched_pp['estate_type'].value_counts(normalize=True)], axis=1, keys=['full_pp', 'unmatched_pp']))

              full_pp  unmatched_pp
estate_type                        
F            0.846589      0.808936
L            0.153411      0.191064


In [214]:
print(pp['price_paid'].describe())
print(unmatched_pp['price_paid'].describe())

count    1.355570e+05
mean     5.126429e+05
std      1.749420e+06
min      1.000000e+02
25%      2.750000e+05
50%      3.650000e+05
75%      5.030000e+05
max      4.141087e+08
Name: price_paid, dtype: float64
count    3.677300e+04
mean     6.629744e+05
std      3.131950e+06
min      1.000000e+02
25%      2.650000e+05
50%      3.700000e+05
75%      5.550000e+05
max      4.141087e+08
Name: price_paid, dtype: float64


- Unmatched set contains slightly more expensive properties

In [215]:
epc_address_cols = ['address1', 'address2', 'address3', 'postcode']
pp_address_cols = ['paon', 'saon', 'street', 'locality', 'postcode']
print(pp[pp_address_cols][(pp['postcode'] == 'RG9 1AF') & (pp['saon'] == '2')])
print(epc[epc_address_cols][(epc['postcode'] == 'RG9 1AF') & (epc['address1'] == '2 PERPETUAL HOUSE')])

                  paon saon        street locality postcode
76650  PERPETUAL HOUSE    2  STATION ROAD      NaN  RG9 1AF
                address1      address2          address3 postcode
93960  2 PERPETUAL HOUSE  STATION ROAD  HENLEY-ON-THAMES  RG9 1AF


- Mismatch appears to be due to inclusion of 'HENLEY-ON-THAMES' in address3 in the EPC (and not in locality in PP)

In [216]:
epc_address_cols = ['address1', 'address2', 'address3', 'postcode']
pp_address_cols = ['paon', 'saon', 'street', 'locality', 'postcode']
print(pp[pp_address_cols][(pp['postcode'] == 'RG9 3JH') & (pp['paon'] == '5A')])
print(epc[epc_address_cols][(epc['postcode'] == 'RG9 3JH') & (epc['address1'] == '5A OAKS ROAD')])

      paon saon     street  locality postcode
79532   5A  NaN  OAKS ROAD  SHIPLAKE  RG9 3JH
            address1  address2          address3 postcode
215444  5A OAKS ROAD  SHIPLAKE  HENLEY-ON-THAMES  RG9 3JH


- Did not match due to inclusion of 'HENLEY-ON-THAMES' in address3 in the EPC (in addition to the Shiplake locality in PP)

#### Third Join (subset matching)

In [217]:
postcode_match = pp.merge(epc, on='postcode', how='inner')

In [218]:

def compare(row):
    return row['address_tokens_x'].issubset(row['address_tokens_y'])

postcode_match['result'] = postcode_match.apply(compare, axis=1)

print(postcode_match.head())

                              unique_id  price_paid  deed_date  postcode property_type_x new_build estate_type          saon                 paon street   locality       town  district       county transaction_category                                    linked_data_uri                                         address_x                                   address_tokens_x        certificate_number                       address1             address2 address3   posttown                                        address_y local_authority local_authority_label built_form           construction_age_band  current_energy_efficiency current_energy_rating  energy_consumption_current  extension_count  floor_height  heating_cost_current inspection_date lodgement_date  number_habitable_rooms  number_heated_rooms  potential_energy_efficiency property_type_y            tenure  total_floor_area transaction_type          uprn      uprn_source                                   address_tokens_y  result
0  EED7

In [219]:
postcode_match['result'].value_counts()

result
False    2769807
True      121525
Name: count, dtype: int64

In [220]:
subset_matches = postcode_match[postcode_match['result']]

print(f"Third join (subset method) gives: {subset_matches['unique_id'].nunique()} matches. A matching rate of {100 * subset_matches['unique_id'].nunique()/pp.shape[0]:.1f}%")

Third join (subset method) gives: 105419 matches. A matching rate of 77.8%


In [221]:
multi_match = subset_matches.groupby('unique_id').size()

print(multi_match[multi_match > 1])

unique_id
00208A94-9E5F-4272-B229-A6B4E4DE4B49    2
0021847D-1FF1-4D1B-85B9-10861F3616C2    2
01975290-00E2-4DD9-AFE5-8F92E562DF73    2
01D08B6C-7DF8-42CA-968E-7B38C3AB248B    2
01EB45EF-73C5-40F3-E063-4704A8C05FDE    3
                                       ..
FFA361DB-8C61-8A03-E053-4804A8C01F61    2
FFA361DB-8CA0-8A03-E053-4804A8C01F61    2
FFA361DB-8CA8-8A03-E053-4804A8C01F61    2
FFA361DB-8CAE-8A03-E053-4804A8C01F61    3
FFA361DB-8CB2-8A03-E053-4804A8C01F61    3
Length: 12514, dtype: int64


In [222]:
multi_match = subset_matches.groupby('unique_id')['uprn'].nunique()

print(multi_match[multi_match > 1])

unique_id
01EB45F0-15D4-40F3-E063-4704A8C05FDE     2
01EB45F0-161B-40F3-E063-4704A8C05FDE     2
01EB45F0-164C-40F3-E063-4704A8C05FDE     4
01EB45F0-1656-40F3-E063-4704A8C05FDE     2
01EB45F0-173C-40F3-E063-4704A8C05FDE     2
                                        ..
FD226036-9C27-4CB7-E053-4804A8C00430     2
FFA361DB-1951-8A03-E053-4804A8C01F61     2
FFA361DB-19E9-8A03-E053-4804A8C01F61     2
FFA361DB-1A71-8A03-E053-4804A8C01F61    43
FFA361DB-1B05-8A03-E053-4804A8C01F61     2
Name: uprn, Length: 975, dtype: int64


In [223]:
print(pp[pp['unique_id'] == '01EB45F0-15D4-40F3-E063-4704A8C05FDE']['address'])
print(epc[epc['postcode'] == 'OX5 1BN'])

29870    4  CHURCHILL ROAD  OX5 1BN
Name: address, dtype: str
              certificate_number               address1          address2          address3 postcode    posttown                                          address local_authority local_authority_label     built_form         construction_age_band  current_energy_efficiency current_energy_rating  energy_consumption_current  extension_count  floor_height  heating_cost_current inspection_date lodgement_date  number_habitable_rooms  number_heated_rooms  potential_energy_efficiency property_type            tenure  total_floor_area   transaction_type          uprn      uprn_source                                     address_tokens
27986   0180-2773-9190-2029-7291       4 CHURCHILL ROAD        KIDLINGTON               NaN  OX5 1BN  KIDLINGTON             4 CHURCHILL ROAD KIDLINGTON  OX5 1BN       E07000177              Cherwell       Detached  England and Wales: 1930-1949                         65                     D              

- 4 Churchill road OX5 1BN subset matched to both correct and 'APARTMENT 4 THE LOFTS 7 CHURCHILL ROAD OX5 1BN'

In [224]:
print(pp[pp['unique_id'] == 'FFA361DB-1A71-8A03-E053-4804A8C01F61']['address'])
print(pp[pp['postcode'] == 'OX2 0FL'])
epc['uprn'] = epc['uprn'].fillna('1').astype('int64')
print(epc[epc['postcode'] == 'OX2 0FL'])

104748    RIVERSIDE COURT 9 9 WEST WAY  OX2 0FL
Name: address, dtype: str
                                   unique_id  price_paid  deed_date postcode property_type new_build estate_type saon               paon    street locality    town             district       county transaction_category                                    linked_data_uri                                 address                                     address_tokens
104713  FFA361DB-1A66-8A03-E053-4804A8C01F61      250000 2022-09-13  OX2 0FL             F         Y           L    1  RIVERSIDE COURT 9  WEST WAY      NaN  OXFORD  VALE OF WHITE HORSE  OXFORDSHIRE                    A  http://landregistry.data.gov.uk/data/ppi/trans...   RIVERSIDE COURT 9 1 WEST WAY  OX2 0FL  frozenset({COURT, RIVERSIDE, 0FL, 1, 9, WEST, ...
104714  FFA361DB-1A61-8A03-E053-4804A8C01F61      250000 2022-08-30  OX2 0FL             F         Y           L   10  RIVERSIDE COURT 9  WEST WAY      NaN  OXFORD  VALE OF WHITE HORSE  OXFORDSHIRE       

- One of the '9's is lost when a token set is made from  RIVERSIDE COURT 9 9 WEST WAY  OX2 0FL. Which then matches with the 44 properties at  'RIVERSIDE COURT 9 West Way OX2 0FL'.

In [225]:
pp['token_count'] = pp['address'].str.count(r"\S+")

print((pp['token_count'] > pp['address_tokens'].apply(len)).sum())

3113


In [226]:
lost_tokens = pp[pp['token_count'] > pp['address_tokens'].apply(len)]
multi_uprn = multi_match[multi_match > 1]
result = lost_tokens[lost_tokens['unique_id'].isin(multi_uprn.index)]
print(f"Of the 975 unique_ids matched to >1 uprn, {result.shape[0]} or {100 * result.shape[0]/ multi_uprn.shape[0]:.1f}% lost duplicate tokens in token set formation.")

Of the 975 unique_ids matched to >1 uprn, 219 or 22.5% lost duplicate tokens in token set formation.


- Needs date filtering

In [227]:
third_join = subset_matches[~subset_matches['unique_id'].isin(multi_uprn.index)]
print(third_join.shape)

(117362, 47)


In [228]:

third_join['time_to_sale'] = third_join['deed_date'] - third_join['inspection_date']

drop_neg = third_join[third_join['time_to_sale'] >= '0 days']
third_join_presale_epc = drop_neg.sort_values(by='time_to_sale', ascending=True).drop_duplicates(subset='unique_id')
print(f"Properties remaining with matched pre-sale EPCs: {third_join_presale_epc.shape[0]}")
print(f"Corresponss to a {100 * third_join_presale_epc.shape[0]/pp.shape[0]:.1f}% match rate")


Properties remaining with matched pre-sale EPCs: 91150
Corresponss to a 67.2% match rate
